# Notebook 14 — Backlog Additions: Bootstrap CIs, Multiple-Comparison Correction, Rubin Pooling, Listwise Baseline, Outcome Validation

**Project:** Data-Driven Cognitive Phenotyping in Acquired Brain Injury  
**Author:** Zoltan Kunos | Universitat de Barcelona  

Implements the five robustness items raised in the methods backlog:

- **A-3** — Bootstrap 95% confidence intervals for the H1–H4 headline metrics.
- **A-2** — Multiple-comparison correction (Holm) over the 45 H2 ARI pairs.
- **A-4** — Rubin-pooled multiple imputation (M=5 MICE draws) for the headline metrics.
- **A-5** — Listwise-deletion baseline as a sanity check on the imputed pipeline.
- **D-1** — Outcome-adjacent validation: time-since-injury (TSI) by phenotype and within-patient phenotype stability across repeat assessments.

Outputs are written to `results/backlog_results.pkl` plus six CSV summaries used by the dashboard's Robustness page and by Chapter 4 §4.12.

## Imports and configuration

In [1]:
from __future__ import annotations
import pickle, time, warnings, json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, silhouette_samples)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import umap
import hdbscan

warnings.filterwarnings("ignore")

RS = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
DATA = ROOT / "data"
IMPUTED = DATA / "imputed_csv"

rng = np.random.default_rng(RS)


def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

## Load shared infrastructure

Reuses the labels and domain scores produced by NB5/NB6 so the backlog metrics are computed against the exact same fitted pipeline reported in Chapter 4.

In [2]:
log("Loading shared infrastructure...")
with open(RESULTS / "shared_infrastructure.pkl", "rb") as f:
    infra = pickle.load(f)
with open(RESULTS / "eda_output.pkl", "rb") as f:
    eda = pickle.load(f)
with open(RESULTS / "fitted_models.pkl", "rb") as f:
    models = pickle.load(f)

ELIGIBLE_VARS = infra["ELIGIBLE_VARS"]
DOMAINS = infra["DOMAINS"]
METHODS = infra["METHODS"]

# Labels from fixed-k k-means (used for H2)
labels_fixed = infra["cluster_labels_fixed"]  # dict: method -> array(22075,)
# Labels from HDBSCAN on MICE (primary phenotypes)
labels_mice = infra["cluster_labels"]["MICE"]

# Domain scores from MICE (for silhouette bootstrap)
domain_scores_mice = np.asarray(infra["domain_scores"]["MICE"])

# Raw dataframe for TSI + patient ID analysis
raw = pd.read_excel(DATA / "0_tests2.xlsx")
log(f"Raw data: {raw.shape}")

[00:23:00] Loading shared infrastructure...


[00:23:01] Raw data: (22075, 31)


## A-3 — Bootstrap 95% CIs for the H1–H4 headline metrics

`B = 500` bootstrap resamples. Silhouette is O(N²), so each draw is subsampled to `SIL_SUB = 5000` points (sklearn's documented practice for large N). The bootstrap CI is the empirical 2.5th–97.5th percentile.

In [3]:
log("=" * 70)
log("A-3: Bootstrap 95% CIs for key metrics")
log("=" * 70)

B = 500        # bootstrap resamples
ALPHA = 0.05

def bci(vals, alpha=ALPHA):
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return float(lo), float(hi)

[00:23:01] ======================================================================


[00:23:01] A-3: Bootstrap 95% CIs for key metrics


[00:23:01] ======================================================================


### H1 — bootstrap silhouette of the MICE/HDBSCAN solution

In [4]:
SIL_SUB = 5000
log(f"Bootstrap H1 silhouette ({B} iters, sample_size={SIL_SUB})...")
t0 = time.time()
ds = domain_scores_mice
lab = labels_mice
sil_boot = []
n = len(lab)
for b in range(B):
    idx = rng.integers(0, n, size=n)
    sub = rng.choice(n, size=SIL_SUB, replace=False)
    boot_ds = ds[idx][sub]
    boot_lab = lab[idx][sub]
    if len(np.unique(boot_lab)) >= 2:
        try:
            sil_boot.append(silhouette_score(boot_ds, boot_lab))
        except Exception:
            pass
sil_boot = np.asarray(sil_boot)
h1_sil_ci = bci(sil_boot)
log(f"  H1 silhouette = {sil_boot.mean():.4f} [{h1_sil_ci[0]:.4f}, {h1_sil_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

[00:23:01] Bootstrap H1 silhouette (500 iters, sample_size=5000)...


[00:24:26]   H1 silhouette = 0.0446 [0.0376, 0.0525]  (85.1s)


### H2 — bootstrap mean ARI plus the 45 pairwise ARIs

In [5]:
log(f"Bootstrap H2 pairwise ARIs ({B} iters)...")
t0 = time.time()
methods = list(labels_fixed.keys())
M = len(methods)
label_arrs = {m: np.asarray(labels_fixed[m]) for m in methods}

pair_boot = {(methods[i], methods[j]): [] for i in range(M) for j in range(i + 1, M)}
mean_boot = []
for b in range(B):
    idx = rng.integers(0, n, size=n)
    ari_b = []
    for (mi, mj), store in pair_boot.items():
        v = adjusted_rand_score(label_arrs[mi][idx], label_arrs[mj][idx])
        store.append(v)
        ari_b.append(v)
    mean_boot.append(float(np.mean(ari_b)))

mean_boot = np.asarray(mean_boot)
h2_mean_ci = bci(mean_boot)
log(f"  H2 mean ARI = {mean_boot.mean():.4f} [{h2_mean_ci[0]:.4f}, {h2_mean_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

h2_pair_ci = {f"{mi}-{mj}": bci(np.asarray(v)) for (mi, mj), v in pair_boot.items()}

[00:24:26] Bootstrap H2 pairwise ARIs (500 iters)...


[00:24:50]   H2 mean ARI = 0.7096 [0.7021, 0.7169]  (23.3s)


### H3 — bootstrap chi² and Cramér's V on cluster × clinical unit

In [6]:
log(f"Bootstrap H3 chi2 + Cramer's V ({B} iters)...")
t0 = time.time()
df_el = eda["df_eligible"].reset_index(drop=True)
unit_col = "C_UNITATMEDICA"
if unit_col in df_el.columns:
    units = df_el[unit_col].values
else:
    units = raw[unit_col].values
mask_unit = pd.notna(units) & (units != 0)
lab_h3 = labels_mice[mask_unit]
unit_h3 = np.asarray(units)[mask_unit]

def chi2_cramers(labels, groups):
    ct = pd.crosstab(labels, groups)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n_h = ct.values.sum()
    k = min(ct.shape) - 1
    V = np.sqrt(chi2 / (n_h * k)) if n_h * k > 0 else np.nan
    return chi2, p, V

c2_boot, cv_boot = [], []
nh = len(lab_h3)
for b in range(B):
    idx = rng.integers(0, nh, size=nh)
    try:
        c2, _, V = chi2_cramers(lab_h3[idx], unit_h3[idx])
        c2_boot.append(c2); cv_boot.append(V)
    except Exception:
        pass
h3_c2_ci = bci(np.asarray(c2_boot))
h3_cv_ci = bci(np.asarray(cv_boot))
log(f"  H3 chi2 ~ {np.mean(c2_boot):.2f} [{h3_c2_ci[0]:.2f}, {h3_c2_ci[1]:.2f}]"
    f"  Cramer's V = {np.mean(cv_boot):.4f} [{h3_cv_ci[0]:.4f}, {h3_cv_ci[1]:.4f}]"
    f"  ({time.time() - t0:.1f}s)")

[00:24:50] Bootstrap H3 chi2 + Cramer's V (500 iters)...


[00:24:51]   H3 chi2 ~ 953.09 [847.68, 1063.97]  Cramer's V = 0.1657 [0.1563, 0.1751]  (0.8s)


### H4 — bootstrap silhouette gap (domain-level minus variable-level)

In [7]:
log(f"Bootstrap H4 silhouette gap ({B // 2} iters, uses KMeans per iter)...")
t0 = time.time()
mice_imputed = pd.read_csv(IMPUTED / "Imputed_MICE.csv")[ELIGIBLE_VARS].values
var_scaler = StandardScaler().fit(mice_imputed)
var_feats = var_scaler.transform(mice_imputed)
dom_feats = StandardScaler().fit_transform(domain_scores_mice)

sil_gap_boot = []
Bh4 = 200
for b in range(Bh4):
    idx = rng.integers(0, n, size=n)
    km_d = KMeans(n_clusters=2, random_state=RS + b, n_init=5).fit(dom_feats[idx])
    km_v = KMeans(n_clusters=2, random_state=RS + b, n_init=5).fit(var_feats[idx])
    sub = rng.choice(n, size=SIL_SUB, replace=False)
    try:
        s_d = silhouette_score(dom_feats[idx][sub], km_d.labels_[sub])
        s_v = silhouette_score(var_feats[idx][sub], km_v.labels_[sub])
        sil_gap_boot.append(s_d - s_v)
    except Exception:
        pass
h4_gap_ci = bci(np.asarray(sil_gap_boot))
log(f"  H4 silhouette gap = {np.mean(sil_gap_boot):.4f}"
    f" [{h4_gap_ci[0]:.4f}, {h4_gap_ci[1]:.4f}]  ({time.time() - t0:.1f}s)")

[00:24:51] Bootstrap H4 silhouette gap (250 iters, uses KMeans per iter)...


[00:30:21]   H4 silhouette gap = -0.0022 [-0.0208, 0.0129]  (330.3s)


## A-2 — Multiple-comparison correction over the 45 H2 ARI pairs

Provisional empirical p-value: proportion of bootstrap samples with `ARI <= 0.5`. Because the empirical p is quantized at `1/(B+1) ≈ 0.002`, no pair can clear the smallest Holm threshold (`0.05/45 ≈ 0.0011`). The script `14b_fix_a2.py` (and the follow-up notebook `14b_Fix_A2_Parametric_pvalues.ipynb`) replaces this with a parametric one-sided p-value that *can* be arbitrarily small, and writes the definitive `Backlog_H2_Holm.csv`.

In [8]:
log("=" * 70)
log("A-2: Multiple-comparison correction for 45 H2 ARI pairs (H0: ARI <= 0.5)")
log("=" * 70)

pair_pvals = {}
for pair_key, ari_samples in pair_boot.items():
    arr = np.asarray(ari_samples)
    p = float((arr <= 0.5).mean())
    p = max(p, 1.0 / (B + 1))
    pair_pvals[f"{pair_key[0]}-{pair_key[1]}"] = p

from statsmodels.stats.multitest import multipletests
names = list(pair_pvals.keys())
pvals = np.asarray([pair_pvals[n_] for n_ in names])
reject, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")
n_pass_holm = int(reject.sum())
log(f"  {n_pass_holm}/45 pairs reject H0 after Holm correction at alpha=0.05")

pair_pval_df = pd.DataFrame({
    "pair": names,
    "bootstrap_p": pvals,
    "holm_adj_p": p_holm,
    "reject_H0_at_0.05": reject,
})
pair_pval_df = pair_pval_df.sort_values("bootstrap_p").reset_index(drop=True)

[00:30:21] ======================================================================


[00:30:21] A-2: Multiple-comparison correction for 45 H2 ARI pairs (H0: ARI <= 0.5)


[00:30:21] ======================================================================


[00:30:21]   0/45 pairs reject H0 after Holm correction at alpha=0.05


## A-4 — Rubin-pooled multiple imputation

Five independent MICE imputations with `sample_posterior=True` are each run through the full pipeline (domain scoring → UMAP → HDBSCAN → KNN noise reassignment), then the headline metrics are pooled by Rubin's rules. Without per-imputation variance estimates we use only the between-imputation variance, which underestimates total variance — see Chapter 5 for caveats.

In [9]:
log("=" * 70)
log("A-4: Rubin-pooled multiple imputation (M=5 MICE draws)")
log("=" * 70)

M_MI = 5
df_el = eda["df_eligible"].reset_index(drop=True)
raw_mat = df_el[ELIGIBLE_VARS].astype(float).values

def run_mice(seed):
    imp = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        sample_posterior=True,
        random_state=seed,
    )
    return imp.fit_transform(raw_mat)

def run_pipeline(imputed_matrix, seed):
    dom_cols = list(DOMAINS.keys())
    dom = np.zeros((imputed_matrix.shape[0], len(dom_cols)))
    miss_rates = df_el[ELIGIBLE_VARS].isna().mean().to_dict()
    for di, dname in enumerate(dom_cols):
        vars_in = [v for v in DOMAINS[dname] if v in ELIGIBLE_VARS]
        cols = [ELIGIBLE_VARS.index(v) for v in vars_in]
        ws = np.asarray([1.0 - miss_rates[v] for v in vars_in])
        ws = ws / ws.sum() if ws.sum() > 0 else ws
        dom[:, di] = imputed_matrix[:, cols] @ ws
    dom_std = StandardScaler().fit_transform(dom)

    reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0,
                        random_state=seed, n_jobs=1)
    emb = reducer.fit_transform(dom_std)
    clu = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50,
                          cluster_selection_method="eom", cluster_selection_epsilon=0.0)
    lbl = clu.fit_predict(emb)
    from sklearn.neighbors import KNeighborsClassifier
    nmask = lbl == -1
    if nmask.any() and (~nmask).any():
        knn = KNeighborsClassifier(n_neighbors=10).fit(emb[~nmask], lbl[~nmask])
        lbl[nmask] = knn.predict(emb[nmask])
    if len(np.unique(lbl)) >= 2:
        sil = silhouette_score(dom_std, lbl)
    else:
        sil = np.nan
    return dom_std, lbl, sil

[00:30:21] ======================================================================


[00:30:21] A-4: Rubin-pooled multiple imputation (M=5 MICE draws)


[00:30:21] ======================================================================


In [10]:
mi_results = []
for m in range(M_MI):
    t0 = time.time()
    log(f"MI {m+1}/{M_MI}: MICE imputation...")
    imputed = run_mice(seed=RS + 1000 + m)
    log(f"MI {m+1}/{M_MI}: UMAP + HDBSCAN...")
    dom_std, lbl, sil = run_pipeline(imputed, seed=RS + 1000 + m)
    mask = pd.notna(df_el[unit_col]) & (df_el[unit_col] != 0)
    lab_h3 = lbl[mask.values]
    unit_h3 = df_el.loc[mask, unit_col].values
    chi2_m, p_m, V_m = chi2_cramers(lab_h3, unit_h3)
    imputed_std = StandardScaler().fit_transform(imputed)
    km_d = KMeans(n_clusters=2, random_state=RS + 1000 + m, n_init=5).fit(dom_std)
    km_v = KMeans(n_clusters=2, random_state=RS + 1000 + m, n_init=5).fit(imputed_std)
    s_d = silhouette_score(dom_std, km_d.labels_)
    s_v = silhouette_score(imputed_std, km_v.labels_)
    gap = s_d - s_v
    n_clusters = len(np.unique(lbl))
    mi_results.append(dict(
        m=m, silhouette=sil, chi2=chi2_m, cramers_v=V_m,
        gap=gap, s_d=s_d, s_v=s_v, n_clusters=n_clusters,
        time_s=time.time() - t0,
    ))
    log(f"MI {m+1}/{M_MI}: sil={sil:.4f} chi2={chi2_m:.2f} V={V_m:.4f}"
        f" gap={gap:.4f}  ({time.time() - t0:.1f}s)")

mi_df = pd.DataFrame(mi_results)
mi_df

[00:30:21] MI 1/5: MICE imputation...


[00:30:21] MI 1/5: UMAP + HDBSCAN...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[00:30:40] MI 1/5: sil=-0.0578 chi2=1068.58 V=0.1433 gap=0.0473  (19.1s)


[00:30:40] MI 2/5: MICE imputation...


...

[00:31:21] MI 4/5: sil=-0.0534 chi2=1795.41 V=0.1439 gap=0.0494  (13.5s)


[00:31:21] MI 5/5: MICE imputation...


[00:31:22] MI 5/5: UMAP + HDBSCAN...


[00:31:34] MI 5/5: sil=0.0009 chi2=1350.27 V=0.1395 gap=0.0507  (12.9s)


,m,silhouette,chi2,cramers_v,gap,s_d,s_v,n_clusters,time_s
0,0,-0.057803,1068.578809,0.143295,0.047298,0.480990,0.433692,4,19.086369
1,1,0.064360,1752.343590,0.129754,0.050420,0.484307,0.433886,7,13.968903
2,2,0.024233,1606.911185,0.215213,0.054033,0.482285,0.428253,3,13.862282
3,3,-0.053421,1795.413347,0.143875,0.049361,0.484090,0.434730,6,13.505533
4,4,0.000918,1350.273010,0.139498,0.050678,0.478507,0.427829,5,12.924335


In [11]:
def rubin_pool(estimates, var_within=None):
    """Rubin's rules for scalar quantities without per-imputation variance
    (i.e., pool the point estimates only, treating them as draws)."""
    est = np.asarray(estimates, dtype=float)
    m = len(est)
    if m < 2:
        return float(est.mean()), 0.0, 0.0, 0.0
    qbar = float(est.mean())
    B_var = float(((est - qbar) ** 2).sum() / (m - 1))
    T = (1.0 + 1.0 / m) * B_var
    se = float(np.sqrt(T))
    from scipy.stats import t as tdist
    df_r = m - 1
    tcrit = tdist.ppf(0.975, df_r)
    return qbar, se, qbar - tcrit * se, qbar + tcrit * se


rubin_rows = []
for col in ("silhouette", "chi2", "cramers_v", "gap"):
    est = mi_df[col].values
    qbar, se, lo, hi = rubin_pool(est)
    rubin_rows.append(dict(
        metric=col, pooled=qbar, se_between=se,
        ci_lo=lo, ci_hi=hi,
        values=",".join(f"{v:.4f}" for v in est),
    ))
rubin_tab = pd.DataFrame(rubin_rows)
log("Rubin-pooled results:")
for r in rubin_rows:
    log(f"  {r['metric']:>12s}: {r['pooled']:.4f}  [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]"
        f"  (se_between={r['se_between']:.4f})")
rubin_tab

[00:31:34] Rubin-pooled results:


[00:31:34]     silhouette: -0.0043  [-0.1626, 0.1539]  (se_between=0.0570)


[00:31:34]           chi2: 1514.7040  [589.8880, 2439.5199]  (se_between=333.0936)


[00:31:34]      cramers_v: 0.1543  [0.0494, 0.2593]  (se_between=0.0378)


[00:31:34]            gap: 0.0504  [0.0429, 0.0578]  (se_between=0.0027)


,metric,pooled,se_between,ci_lo,ci_hi,values
0,silhouette,-0.004343,0.057003,-0.162608,0.153922,"-0.0578,0.0644,0.0242,-0.0534,0.0009"
1,chi2,1514.703988,333.093552,589.888028,2439.519949,"1068.5788,1752.3436,1606.9112,1795.4133,1350.2730"
2,cramers_v,0.154327,0.037796,0.049389,0.259266,"0.1433,0.1298,0.2152,0.1439,0.1395"
3,gap,0.050358,0.002681,0.042913,0.057803,"0.0473,0.0504,0.0540,0.0494,0.0507"


## A-5 — Listwise-deletion baseline

Refits the full pipeline on the complete-case subset (no imputation at all) and computes the agreement (ARI/NMI) with the imputed labels on the shared patients. This isolates whether the phenotype structure is an artefact of imputation or is already visible in the observed data.

In [12]:
log("=" * 70)
log("A-5: Listwise-deletion baseline (complete cases only)")
log("=" * 70)

cc_mask = df_el[ELIGIBLE_VARS].notna().all(axis=1).values
cc_data = df_el.loc[cc_mask, ELIGIBLE_VARS].astype(float).values
cc_count = int(cc_mask.sum())
log(f"Complete cases: {cc_count} / {len(df_el)} = {cc_count / len(df_el):.3%}")

dom_cols = list(DOMAINS.keys())
dom_cc = np.zeros((cc_count, len(dom_cols)))
miss_rates = df_el[ELIGIBLE_VARS].isna().mean().to_dict()
for di, dname in enumerate(dom_cols):
    vars_in = [v for v in DOMAINS[dname] if v in ELIGIBLE_VARS]
    cols = [ELIGIBLE_VARS.index(v) for v in vars_in]
    ws = np.asarray([1.0 - miss_rates[v] for v in vars_in])
    ws = ws / ws.sum() if ws.sum() > 0 else ws
    dom_cc[:, di] = cc_data[:, cols] @ ws
dom_cc_std = StandardScaler().fit_transform(dom_cc)

reducer_cc = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0,
                       random_state=RS, n_jobs=1)
emb_cc = reducer_cc.fit_transform(dom_cc_std)
clu_cc = hdbscan.HDBSCAN(min_cluster_size=500, min_samples=50,
                         cluster_selection_method="eom", cluster_selection_epsilon=0.0)
lbl_cc_raw = clu_cc.fit_predict(emb_cc)
from sklearn.neighbors import KNeighborsClassifier
nmask = lbl_cc_raw == -1
lbl_cc = lbl_cc_raw.copy()
if nmask.any() and (~nmask).any():
    knn = KNeighborsClassifier(n_neighbors=10).fit(emb_cc[~nmask], lbl_cc[~nmask])
    lbl_cc[nmask] = knn.predict(emb_cc[nmask])

n_cc_clusters = len(np.unique(lbl_cc))
sil_cc = (silhouette_score(dom_cc_std, lbl_cc)
          if n_cc_clusters >= 2 else np.nan)
noise_pre_knn_cc = float((lbl_cc_raw == -1).mean())

mice_on_cc = labels_mice[cc_mask]
ari_cc_vs_mice = adjusted_rand_score(mice_on_cc, lbl_cc)
nmi_cc_vs_mice = normalized_mutual_info_score(mice_on_cc, lbl_cc)
log(f"  Listwise clusters: {n_cc_clusters},  silhouette = {sil_cc:.4f}"
    f",  pre-KNN noise = {noise_pre_knn_cc:.3%}")
log(f"  ARI(listwise vs MICE) on the {cc_count} shared patients = {ari_cc_vs_mice:.4f}")
log(f"  NMI(listwise vs MICE) on the {cc_count} shared patients = {nmi_cc_vs_mice:.4f}")

ct = pd.crosstab(pd.Series(mice_on_cc, name="MICE_full_22075"),
                 pd.Series(lbl_cc, name="Listwise_9245"))

listwise_tab = pd.DataFrame(dict(
    metric=["n_complete_cases", "fraction_of_full_sample",
            "n_clusters_listwise", "silhouette_listwise",
            "noise_pre_knn_listwise",
            "ARI_listwise_vs_MICE", "NMI_listwise_vs_MICE"],
    value=[cc_count, cc_count / len(df_el),
           n_cc_clusters, sil_cc, noise_pre_knn_cc,
           ari_cc_vs_mice, nmi_cc_vs_mice],
))
listwise_tab

[00:31:34] ======================================================================


[00:31:34] A-5: Listwise-deletion baseline (complete cases only)


[00:31:34] ======================================================================


[00:31:34] Complete cases: 9245 / 17406 = 53.114%


[00:31:41]   Listwise clusters: 2,  silhouette = 0.2571,  pre-KNN noise = 0.000%


[00:31:41]   ARI(listwise vs MICE) on the 9245 shared patients = 0.6249


[00:31:41]   NMI(listwise vs MICE) on the 9245 shared patients = 0.5186


,metric,value
0,n_complete_cases,9245.000000
1,fraction_of_full_sample,0.531139
2,n_clusters_listwise,2.000000
3,silhouette_listwise,0.257060
4,noise_pre_knn_listwise,0.000000
5,ARI_listwise_vs_MICE,0.624898
6,NMI_listwise_vs_MICE,0.518585


## D-1 — Outcome-adjacent validation (TSI and within-patient stability)

Two checks that the phenotype split is clinically interpretable:

1. Time since injury (TSI) at assessment, contrasted between phenotypes via Mann–Whitney U.
2. Within-patient phenotype stability for patients with ≥2 assessments — the fraction whose phenotype is consistent across visits, plus the asymmetry of transitions for two-visit patients (0→1 vs 1→0).

In [13]:
log("=" * 70)
log("D-1: Outcome-adjacent validation")
log("=" * 70)

tsi_col = "TSI_TO_ASSESSMENT_DAYS"
tsi_vals = df_el[tsi_col].values if tsi_col in df_el.columns else raw[tsi_col].values
tsi_df = pd.DataFrame({"cluster": labels_mice, "TSI_days": tsi_vals})
tsi_desc = tsi_df.groupby("cluster")["TSI_days"].describe()
# k-agnostic: Kruskal-Wallis across all severity tiers (u_p kept for downstream save)
_groups = [g["TSI_days"].dropna().values for _, g in tsi_df.groupby("cluster")]
u_stat, u_p = stats.kruskal(*_groups)
_medians = tsi_df.groupby("cluster")["TSI_days"].median().round(0).to_dict()
log(f"  TSI by tier (days): medians={_medians}, Kruskal-Wallis p={u_p:.3e}")
log("  TSI descriptives by cluster:\n" + tsi_desc.to_string())
tsi_desc

[00:31:41] ======================================================================


[00:31:41] D-1: Outcome-adjacent validation


[00:31:41] ======================================================================


[00:31:41]   TSI by tier (days): medians={0: 214.0, 1: 172.0, 2: 188.0}, Kruskal-Wallis p=4.047e-28


[00:31:41]   TSI descriptives by cluster:
          count         mean          std       min         25%         50%         75%           max
cluster                                                                                              
0        6725.0  1045.615953  2119.958818  0.719595  108.544479  214.470336  822.525324  22096.511030
1        4353.0   751.727018  1755.099862  1.473657   89.675822  171.559340  424.765197  16851.499873
2        6328.0   820.710725  1812.528539  0.934444   93.646736  187.653096  558.842329  21352.616366


,count,mean,std,min,25%,50%,75%,max
cluster,,,,,,,,
0,6725.0,1045.615953,2119.958818,0.719595,108.544479,214.470336,822.525324,22096.511030
1,4353.0,751.727018,1755.099862,1.473657,89.675822,171.559340,424.765197,16851.499873
2,6328.0,820.710725,1812.528539,0.934444,93.646736,187.653096,558.842329,21352.616366


In [14]:
id_col = "ID"
pat_ids = df_el[id_col].values if id_col in df_el.columns else raw[id_col].values
wp_df = pd.DataFrame({"patient_id": pat_ids, "cluster": labels_mice, "tsi": tsi_vals})
counts = wp_df.groupby("patient_id").size()
multi = counts[counts >= 2].index
log(f"  {len(multi)} patients have >=2 assessments (covering {counts[multi].sum()} records)")

patient_stability = (
    wp_df[wp_df.patient_id.isin(multi)]
    .groupby("patient_id")["cluster"]
    .agg(lambda s: s.nunique())
    .reset_index(name="distinct_clusters")
)
stable = (patient_stability.distinct_clusters == 1).sum()
transition = (patient_stability.distinct_clusters > 1).sum()
log(f"  Stable (same phenotype across all assessments): {stable}/{len(multi)}"
    f" = {stable/len(multi):.1%}")
log(f"  Transitioning (>=2 distinct phenotypes): {transition}/{len(multi)}"
    f" = {transition/len(multi):.1%}")

multi2 = counts[counts == 2].index
wp2 = wp_df[wp_df.patient_id.isin(multi2)].sort_values(["patient_id", "tsi"])
transitions = []
for pid, grp in wp2.groupby("patient_id"):
    if grp.cluster.nunique() > 1:
        lbls = grp.cluster.values
        transitions.append((pid, int(lbls[0]), int(lbls[-1])))
tdf = pd.DataFrame(transitions, columns=["patient_id", "from_cluster", "to_cluster"])
if len(tdf):
    tmat = pd.crosstab(tdf.from_cluster, tdf.to_cluster)
    log("  Two-assessment transition matrix (rows=earlier, cols=later):\n" + tmat.to_string())

[00:31:41]   5206 patients have >=2 assessments (covering 15327 records)


[00:31:41]   Stable (same phenotype across all assessments): 2613/5206 = 50.2%


[00:31:41]   Transitioning (>=2 distinct phenotypes): 2593/5206 = 49.8%


[00:31:41]   Two-assessment transition matrix (rows=earlier, cols=later):
to_cluster      0   1    2
from_cluster              
0               0  18   99
1             123   0  240
2             343  89    0


## Persist results

Writes the consolidated `backlog_results.pkl` and six CSV summaries used downstream by the dashboard's Robustness page and by the report's §4.12.

In [15]:
log("=" * 70)
log("Saving...")
log("=" * 70)

backlog = dict(
    A3=dict(
        H1_silhouette=dict(point=float(sil_boot.mean()), ci=h1_sil_ci, B=B),
        H2_mean_ARI=dict(point=float(mean_boot.mean()), ci=h2_mean_ci, B=B),
        H2_pair_CIs=h2_pair_ci,
        H3_chi2=dict(point=float(np.mean(c2_boot)), ci=h3_c2_ci, B=B),
        H3_cramers_v=dict(point=float(np.mean(cv_boot)), ci=h3_cv_ci, B=B),
        H4_silhouette_gap=dict(point=float(np.mean(sil_gap_boot)), ci=h4_gap_ci, B=Bh4),
    ),
    A2=dict(
        n_pairs=45, n_reject_at_0p05=n_pass_holm,
        alpha=0.05, method="Holm-Bonferroni",
        threshold_ARI=0.5,
        pair_pvals={k: float(v) for k, v in pair_pvals.items()},
    ),
    A4=dict(
        M=M_MI, per_imputation=mi_results,
        pooled=rubin_rows,
    ),
    A5=dict(
        n_complete_cases=cc_count,
        fraction=cc_count / len(df_el),
        n_clusters_listwise=n_cc_clusters,
        silhouette_listwise=float(sil_cc),
        noise_pre_knn=noise_pre_knn_cc,
        ARI_listwise_vs_MICE=float(ari_cc_vs_mice),
        NMI_listwise_vs_MICE=float(nmi_cc_vs_mice),
        crosstab=ct.to_dict(),
    ),
    D1=dict(
        TSI_by_cluster=tsi_desc.to_dict(),
        mannwhitney_U=float(u_stat),
        mannwhitney_p=float(u_p),
        multi_assessment_patients=int(len(multi)),
        stable_count=int(stable),
        transition_count=int(transition),
        stable_fraction=float(stable / max(len(multi), 1)),
        two_assessment_transitions=tdf.to_dict("records") if len(tdf) else [],
    ),
)

with open(RESULTS / "backlog_results.pkl", "wb") as f:
    pickle.dump(backlog, f)

pd.DataFrame([
    dict(metric="H1_silhouette",    point=float(sil_boot.mean()),
         ci_lo=h1_sil_ci[0], ci_hi=h1_sil_ci[1]),
    dict(metric="H2_mean_ARI",      point=float(mean_boot.mean()),
         ci_lo=h2_mean_ci[0], ci_hi=h2_mean_ci[1]),
    dict(metric="H3_chi2",          point=float(np.mean(c2_boot)),
         ci_lo=h3_c2_ci[0], ci_hi=h3_c2_ci[1]),
    dict(metric="H3_cramers_v",     point=float(np.mean(cv_boot)),
         ci_lo=h3_cv_ci[0], ci_hi=h3_cv_ci[1]),
    dict(metric="H4_silhouette_gap", point=float(np.mean(sil_gap_boot)),
         ci_lo=h4_gap_ci[0], ci_hi=h4_gap_ci[1]),
]).to_csv(RESULTS / "Backlog_Bootstrap_CIs.csv", index=False)

pair_pval_df.to_csv(RESULTS / "Backlog_H2_Holm.csv", index=False)
rubin_tab.to_csv(RESULTS / "Backlog_Rubin_Pooled.csv", index=False)
listwise_tab.to_csv(RESULTS / "Backlog_Listwise_Comparison.csv", index=False)
tsi_desc.reset_index().to_csv(RESULTS / "Backlog_TSI_Phenotype.csv", index=False)

pd.DataFrame(dict(
    metric=["multi_assessment_patients", "stable", "transitioning",
            "stable_fraction", "mannwhitney_p_TSI"],
    value=[len(multi), stable, transition, stable / max(len(multi), 1), float(u_p)],
)).to_csv(RESULTS / "Backlog_Within_Patient_Transitions.csv", index=False)

log("DONE. Saved results/backlog_results.pkl + 6 CSVs")

[00:31:41] ======================================================================


[00:31:41] Saving...


[00:31:41] ======================================================================


[00:31:41] DONE. Saved results/backlog_results.pkl + 6 CSVs


Run `14b_Fix_A2_Parametric_pvalues.ipynb` next to replace the quantized empirical p-values for A-2 with parametric one-sided p-values, regenerate `Backlog_H2_Holm.csv`, and update `backlog_results.pkl` in place.